
# Regression with an Abalone Dataset — Nonlinear Regression Models

## Assignment

Continue competing in the Kaggle **Regression with an Abalone Dataset** competition as a late submission. Build two regression models using nonlinear methods discussed in the weekly lecture and the ISLR/ISLP text. Interpret the models, investigate assumptions, interpret findings, and generate a Kaggle submission file.

## Nonlinear methods used

This notebook uses two nonlinear methods from the week's topic **Beyond Linearity**:

1. **Polynomial regression with Ridge regularization**  
   Polynomial regression adds powers and interactions of predictors so the fitted model can bend while still remaining linear in the coefficients.

2. **Cubic spline regression with Ridge regularization**  
   Splines divide a predictor into regions and fit smooth piecewise polynomial functions, allowing the relationship between predictors and `Rings` to vary across the predictor range.

The target is `Rings`, which approximates abalone age. Because Kaggle evaluates this competition with RMSLE, the models are trained on `log1p(Rings)` and transformed back to the original scale with `expm1`.



## References

James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An introduction to statistical learning: With applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The elements of statistical learning: Data mining, inference, and prediction* (2nd ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.



## 1. Import libraries


In [ ]:

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures, SplineTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42



## 2. Load Kaggle data

Run this notebook in Kaggle or place `train.csv`, `test.csv`, and optionally `sample_submission.csv` in the same folder as the notebook.

For Kaggle Playground Series Season 4 Episode 4, the default Kaggle input path is usually:

`/kaggle/input/playground-series-s4e4/`


In [ ]:

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SAMPLE_PATH = "sample_submission.csv"

if not os.path.exists(TRAIN_PATH) and os.path.exists("/kaggle/input/playground-series-s4e4/train.csv"):
    TRAIN_PATH = "/kaggle/input/playground-series-s4e4/train.csv"
    TEST_PATH = "/kaggle/input/playground-series-s4e4/test.csv"
    SAMPLE_PATH = "/kaggle/input/playground-series-s4e4/sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
display(train.head())
display(test.head())



## 3. Data quality review

This section checks missing values, duplicated rows, data types, and the basic distribution of the variables.


In [ ]:

print("Data types")
display(train.dtypes)

print("Missing values in train")
display(train.isna().sum())

print("Missing values in test")
display(test.isna().sum())

print("Duplicate rows in train:", train.duplicated().sum())

display(train.describe(include="all").T)



## 4. Define target, predictors, and metric

The response variable is `Rings`. The `id` column is excluded from modeling but retained for the Kaggle submission.


In [ ]:

target = "Rings"
id_col = "id" if "id" in train.columns else None

X_full = train.drop(columns=[target])
y_full = train[target].astype(float)

if id_col:
    X_model = X_full.drop(columns=[id_col])
    test_model = test.drop(columns=[id_col])
else:
    X_model = X_full.copy()
    test_model = test.copy()

numeric_features = X_model.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_model.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

def metrics_table(y_true, y_pred, model_name):
    y_pred = np.clip(y_pred, 0, None)
    return {
        "Model": model_name,
        "RMSLE": rmsle(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }



## 5. Exploratory data analysis

The plots below investigate whether the predictor-response relationships are likely to be nonlinear. Curves, leveling off, or changing slopes justify methods such as polynomial regression and splines.


In [ ]:

plt.figure(figsize=(7, 5))
plt.hist(y_full, bins=35)
plt.xlabel("Rings")
plt.ylabel("Count")
plt.title("Distribution of Rings")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 5))
plt.hist(np.log1p(y_full), bins=35)
plt.xlabel("log1p(Rings)")
plt.ylabel("Count")
plt.title("Distribution of log1p(Rings)")
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# Scatterplots for numeric features
for col in numeric_features:
    plt.figure(figsize=(6.5, 4.5))
    plt.scatter(X_model[col], y_full, alpha=0.25)
    plt.xlabel(col)
    plt.ylabel("Rings")
    plt.title(f"Rings vs. {col}")
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:

# Correlation table
corr = pd.concat([X_model[numeric_features], y_full], axis=1).corr(numeric_only=True)[[target]].drop(index=target)
corr = corr.sort_values(target, ascending=False)
display(corr)

plt.figure(figsize=(8, 5))
plt.bar(corr.index, corr[target])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Correlation with Rings")
plt.title("Numeric Predictor Correlation with Rings")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()



## 6. Train-validation split

The models are trained on `log1p(Rings)` because the competition metric is RMSLE. This transformation also helps reduce right skew and unstable residual variance.


In [ ]:

X_train, X_valid, y_train, y_valid = train_test_split(
    X_model, y_full, test_size=0.20, random_state=RANDOM_STATE
)

y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_valid.shape[0])



## 7. Baseline model

The baseline predicts the mean of `log1p(Rings)`. The nonlinear models should improve on this RMSLE.


In [ ]:

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train_log)
baseline_pred = np.expm1(baseline.predict(X_valid))

baseline_result = metrics_table(y_valid, baseline_pred, "Baseline mean")
pd.DataFrame([baseline_result])



## 8. Model 1 — Polynomial Ridge regression

Polynomial regression allows nonlinear curvature by adding squared, cubic, and interaction terms. Ridge regularization is used because polynomial terms can be highly correlated, and regularization reduces overfitting.


In [ ]:

poly_numeric = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(include_bias=False))
])

poly_preprocess = ColumnTransformer(
    transformers=[
        ("num", poly_numeric, numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="drop"
)

poly_pipeline = Pipeline(steps=[
    ("preprocess", poly_preprocess),
    ("model", Ridge())
])

poly_grid = {
    "preprocess__num__poly__degree": [2, 3],
    "model__alpha": [0.1, 1, 10, 30, 100, 300]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

poly_search = GridSearchCV(
    poly_pipeline,
    param_grid=poly_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    refit=True
)

poly_search.fit(X_train, y_train_log)

poly_pred_log = poly_search.predict(X_valid)
poly_pred = np.clip(np.expm1(poly_pred_log), 0, None)

poly_result = metrics_table(y_valid, poly_pred, "Polynomial Ridge")

print("Best Polynomial Ridge parameters:")
print(poly_search.best_params_)
display(pd.DataFrame([baseline_result, poly_result]))



### Polynomial Ridge interpretation

The polynomial model should be interpreted as a flexible regression surface rather than a single-slope linear model. A numeric predictor can affect `Rings` through its linear term, its squared or cubic term, and interactions with other measurements. Therefore, the effect of a predictor such as weight or shell weight can change as the predictor becomes larger. This is appropriate for biological growth because age often increases with body size, but the rate of increase is not constant over the full size range.



## 9. Model 2 — Cubic Spline Ridge regression

Spline regression uses piecewise polynomial basis functions. This allows local bends in the relationship between predictors and `Rings` while keeping the fitted function smooth.


In [ ]:

spline_numeric = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("spline", SplineTransformer(degree=3, include_bias=False))
])

spline_preprocess = ColumnTransformer(
    transformers=[
        ("num", spline_numeric, numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="drop"
)

spline_pipeline = Pipeline(steps=[
    ("preprocess", spline_preprocess),
    ("model", Ridge())
])

spline_grid = {
    "preprocess__num__spline__n_knots": [4, 5, 6, 7, 8],
    "model__alpha": [0.1, 1, 10, 30, 100, 300]
}

spline_search = GridSearchCV(
    spline_pipeline,
    param_grid=spline_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    refit=True
)

spline_search.fit(X_train, y_train_log)

spline_pred_log = spline_search.predict(X_valid)
spline_pred = np.clip(np.expm1(spline_pred_log), 0, None)

spline_result = metrics_table(y_valid, spline_pred, "Cubic Spline Ridge")

print("Best Cubic Spline Ridge parameters:")
print(spline_search.best_params_)
display(pd.DataFrame([baseline_result, poly_result, spline_result]))



### Cubic Spline Ridge interpretation

The spline model is interpreted as a smooth, region-specific nonlinear model. Instead of forcing a single polynomial shape across the entire predictor range, it allows different local slopes in different regions. For abalone growth, this is useful because small, medium, and large abalones may show different age-size patterns. Ridge regularization controls the complexity of the spline basis so the model does not overreact to local noise.



## 10. Model comparison

The best model is selected using validation RMSLE because this matches the Kaggle competition objective.


In [ ]:

results = pd.DataFrame([baseline_result, poly_result, spline_result]).sort_values("RMSLE")
display(results)

best_model_name = results.iloc[0]["Model"]

if best_model_name == "Polynomial Ridge":
    best_model = poly_search.best_estimator_
    best_pred = poly_pred
elif best_model_name == "Cubic Spline Ridge":
    best_model = spline_search.best_estimator_
    best_pred = spline_pred
else:
    best_model = baseline
    best_pred = baseline_pred

print("Selected model:", best_model_name)



## 11. Diagnostic assumption checks

For these regression models, the key assumptions/diagnostics are:

1. **Functional form:** Nonlinear terms should reduce systematic residual patterns.
2. **Residual center:** Residuals should be centered around zero.
3. **Constant variance:** Residual spread should be reasonably stable across fitted values.
4. **Normality:** Residuals should be approximately normal when inference is emphasized, though Kaggle prediction performance is the main objective.
5. **Independence:** Observations are assumed to be independent rows.
6. **Multicollinearity:** Polynomial and spline bases can create correlated predictors, which is why Ridge regularization is used.


In [ ]:

def diagnostic_plots(y_true, y_pred, model_label):
    residuals = y_true - y_pred
    fitted = y_pred

    plt.figure(figsize=(7, 5))
    plt.scatter(fitted, residuals, alpha=0.30)
    plt.axhline(0, linewidth=2)
    plt.xlabel("Fitted values")
    plt.ylabel("Residuals")
    plt.title(f"Residuals vs. Fitted: {model_label}")
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.scatter(y_true, y_pred, alpha=0.30)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    plt.plot([lo, hi], [lo, hi], linewidth=2)
    plt.xlabel("Actual Rings")
    plt.ylabel("Predicted Rings")
    plt.title(f"Actual vs. Predicted: {model_label}")
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.hist(residuals, bins=35)
    plt.xlabel("Residual")
    plt.ylabel("Count")
    plt.title(f"Residual Distribution: {model_label}")
    plt.grid(alpha=0.3)
    plt.show()

    sm.qqplot(residuals, line="45", fit=True)
    plt.title(f"Q-Q Plot of Residuals: {model_label}")
    plt.grid(alpha=0.3)
    plt.show()

    print(f"Residual summary for {model_label}")
    display(pd.Series(residuals).describe())

    print("Shapiro-Wilk normality test on a sample of residuals")
    sample_resid = pd.Series(residuals).sample(min(5000, len(residuals)), random_state=RANDOM_STATE)
    shapiro_stat, shapiro_p = stats.shapiro(sample_resid)
    print(f"Shapiro statistic: {shapiro_stat:.4f}, p-value: {shapiro_p:.4g}")

    print("Durbin-Watson statistic")
    print(f"{durbin_watson(residuals):.4f}")

diagnostic_plots(y_valid, poly_pred, "Polynomial Ridge")
diagnostic_plots(y_valid, spline_pred, "Cubic Spline Ridge")



### Diagnostic interpretation

Use the residual-vs-fitted plot to check whether the nonlinear model still misses structure. If residuals are mostly scattered around zero without a strong curve, the nonlinear model has captured much of the functional form. If residual spread increases for larger predicted values, that suggests heteroscedasticity. The log transformation helps reduce this problem and aligns the training target with RMSLE. The Q-Q plot and Shapiro-Wilk test assess residual normality, but normality is less critical here than predictive validation because the goal is Kaggle performance rather than classical inference.



## 12. Feature importance

Permutation importance shows how much validation performance worsens when a feature is randomly shuffled. Larger values indicate stronger predictive contribution.


In [ ]:

def neg_rmsle_scorer(estimator, X_eval, y_eval):
    pred_log = estimator.predict(X_eval)
    pred = np.clip(np.expm1(pred_log), 0, None)
    return -rmsle(y_eval, pred)

importance_outputs = {}

for label, estimator in [
    ("Polynomial Ridge", poly_search.best_estimator_),
    ("Cubic Spline Ridge", spline_search.best_estimator_)
]:
    perm = permutation_importance(
        estimator,
        X_valid,
        y_valid,
        scoring=neg_rmsle_scorer,
        n_repeats=8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    imp = pd.DataFrame({
        "Feature": X_valid.columns,
        "Importance_Mean": perm.importances_mean,
        "Importance_Std": perm.importances_std
    }).sort_values("Importance_Mean", ascending=False)

    importance_outputs[label] = imp

    print(label)
    display(imp)

    plt.figure(figsize=(8, 5))
    plt.bar(imp["Feature"], imp["Importance_Mean"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Decrease in negative RMSLE when shuffled")
    plt.title(f"Permutation Importance: {label}")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()



## 13. Partial dependence interpretation

Partial dependence plots show how the fitted model changes as one predictor changes while averaging over the other predictors. These plots help interpret nonlinear effects.


In [ ]:

# Choose key numeric predictors if present.
preferred_features = ["Shell Weight", "Whole Weight", "Length", "Diameter", "Height",
                      "Shell weight", "Whole weight"]

pdp_features = []
for f in preferred_features:
    if f in X_model.columns and f not in pdp_features:
        pdp_features.append(f)

if len(pdp_features) == 0:
    pdp_features = numeric_features[:4]

pdp_features = pdp_features[:4]
print("PDP features:", pdp_features)

for feature in pdp_features:
    fig, ax = plt.subplots(figsize=(7, 5))
    PartialDependenceDisplay.from_estimator(
        best_model,
        X_valid,
        features=[feature],
        ax=ax
    )
    plt.title(f"Partial Dependence of Rings on {feature}: {best_model_name}")
    plt.grid(alpha=0.3)
    plt.show()



## 14. Findings and interpretation

The results should be interpreted in terms of both predictive accuracy and model diagnostics. The baseline model provides a weak reference because it ignores all abalone measurements. The polynomial Ridge model improves by allowing global nonlinear curvature and interactions among physical measurements. The spline Ridge model improves by allowing local nonlinear bends across different regions of each predictor. If the spline model has the lowest validation RMSLE, it suggests that the abalone age-size relationship varies across size ranges and is better captured by piecewise smooth functions. If the polynomial model wins, it suggests that broad curvature and interactions are sufficient for this data set.

The most important predictors are expected to be size and weight variables, especially shell weight, whole weight, diameter, and length when they are available in the Kaggle data. These variables are biologically meaningful because older abalones generally become larger and heavier. However, the partial dependence plots should not be interpreted as purely causal effects because the physical measurements are correlated with one another.

Assumption diagnostics are mixed by design. A perfectly linear residual pattern is unlikely because biological growth data often contain nonlinear and unequal-variance patterns. The log transformation and nonlinear basis functions help address these issues. Ridge regularization also addresses multicollinearity introduced by polynomial and spline features. Overall, model quality is primarily judged by validation RMSLE, but the residual and partial dependence plots provide evidence that nonlinear regression is appropriate for this competition.



## 15. Create Kaggle submission

The selected model is refit on all training rows and used to predict the Kaggle test set. The output file is `submission.csv`.


In [ ]:

# Refit the selected model on all available training data
if best_model_name == "Polynomial Ridge":
    final_model = poly_search.best_estimator_
elif best_model_name == "Cubic Spline Ridge":
    final_model = spline_search.best_estimator_
else:
    final_model = baseline

final_model.fit(X_model, np.log1p(y_full))

test_pred_log = final_model.predict(test_model)
test_pred = np.clip(np.expm1(test_pred_log), 0, None)

submission = pd.DataFrame({
    "id": test[id_col] if id_col else np.arange(len(test)),
    "Rings": test_pred
})

submission.to_csv("submission.csv", index=False)
display(submission.head())
print("Saved submission.csv")



## Final conclusion

This analysis used two nonlinear regression methods, Polynomial Ridge and Cubic Spline Ridge, to model `Rings` in the Kaggle Abalone regression competition. Both methods are appropriate because exploratory plots and biological reasoning suggest that abalone age does not increase linearly with every physical measurement. Polynomial regression captures global curvature and interactions, while spline regression captures smoother local changes across measurement ranges. The final model should be selected based on validation RMSLE, then submitted to Kaggle using the generated `submission.csv`. The diagnostic plots should be used to verify whether residual structure remains; if it does, future improvements could include generalized additive models, gradient boosting, model ensembling, or additional feature engineering such as body-density ratios and measurement interactions.
